# Last-token trigram entropy: numerical check of the formula

We study $H(C\mid A,B)$ for the final three terminals of depth-3 random grammars, in **nats**, without dividing by $\log V$. The grammar mixes binary and ternary rules with `eta_generation = V`. Sentence lengths therefore vary.

First we retain the original sample-based estimate for one grammar. Then we compute the final-trigram probabilities exactly for multiple $(V,F)$ and grammar seeds and compare them with
$$H_{\mathrm{formula}}=\log(1+VF)+(1-F)\log 2.$$

In [ ]:
import math
import random
import numpy as np

from Inside_outside import (
    generate_grammar,
    generate_rules_double_new,
    sample_from_grammar,
)

In [ ]:
# Grammar and sampling settings
V = 16
F = 1 / 4
DEPTH = 3
RULE_SEED = 12345678
SAMPLE_SEED = 20260824
ETA_GENERATION = V
N_SAMPLES = 50_000

m_2 = int(F * V)
m_3 = int(F * V**2)
p_3 = 1 / (m_2 * ETA_GENERATION + m_3)
p_2 = ETA_GENERATION * p_3
p_2, p_3

In [ ]:
# Generate the fixed grammar, then use a separate reproducible sampling seed.
generation_rules, _ = generate_rules_double_new(
    V, V, m_2, m_3, p_2, p_3, DEPTH, RULE_SEED
)
grammar = generate_grammar(generation_rules)

random.seed(SAMPLE_SEED)
triples = np.empty((N_SAMPLES, 3), dtype=np.int64)
lengths = np.empty(N_SAMPLES, dtype=np.int64)

for sample_idx in range(N_SAMPLES):
    start_symbol = random.randint(1, V)
    sequence, _ = sample_from_grammar(
        grammar, str(start_symbol), DEPTH, p_2, p_3
    )
    lengths[sample_idx] = len(sequence)
    triples[sample_idx] = [(int(token) - 1) % V for token in sequence[-3:]]

print(f'Samples: {N_SAMPLES:,}')
print('Sentence lengths and counts:', dict(zip(*np.unique(lengths, return_counts=True))))
print('First five zero-based final triples:', triples[:5])

## Direct conditional-entropy estimate

For each context $(a,b)$, count its possible final tokens $c$. The reported conditional entropy is

$$H(C\mid A,B)=\sum_{a,b} \hat p(a,b) H(\hat p(C\mid a,b)).$$

This is the entropy/loss of the empirical trigram distribution evaluated on the same distribution. The corresponding deterministic trigram classifier predicts $\arg\max_c \hat p(c\mid a,b)$; its plug-in top-1 accuracy is $\sum_{a,b}\hat p(a,b)\max_c\hat p(c\mid a,b)$.

In [ ]:
def trigram_counts(final_triples, vocabulary_size=V):
    counts = np.zeros((vocabulary_size, vocabulary_size, vocabulary_size), dtype=np.int64)
    np.add.at(counts, tuple(final_triples.T), 1)
    return counts


def conditional_entropy_from_counts(counts):
    context_counts = counts.sum(axis=2, keepdims=True)
    probabilities = np.divide(
        counts, context_counts,
        out=np.zeros_like(counts, dtype=float),
        where=context_counts > 0,
    )
    log_probabilities = np.zeros_like(probabilities)
    np.log(probabilities, out=log_probabilities, where=probabilities > 0)
    context_entropies = -(probabilities * log_probabilities).sum(axis=2)
    context_weights = context_counts[..., 0] / counts.sum()
    return float((context_weights * context_entropies).sum())


counts = trigram_counts(triples)
entropy_nats = conditional_entropy_from_counts(counts)
entropy_bits = entropy_nats / math.log(2)
entropy_normalized = entropy_nats / math.log(V)
trigram_accuracy = counts.max(axis=2).sum() / counts.sum()

print(f'Empirical H(last | previous two): {entropy_nats:.6f} nats')
print(f'                                 {entropy_bits:.6f} bits')
print(f'                                 {entropy_normalized:.6f} normalized by log(V)')
print(f'Empirical trigram top-1 accuracy: {trigram_accuracy:.6f} ({trigram_accuracy:.3%})')
print(f'Uniform-guess accuracy:            {1 / V:.6f} ({1 / V:.3%})')
print(f'Observed contexts: {(counts.sum(axis=2) > 0).sum()} / {V**2}')

In [ ]:
entropy_nats

In [ ]:
np.log1p(V*F)+(1-F)*np.log(2)

## Exact numerical comparison across grammars

The recursion below marginalizes over productions, starting from the terminal layer. It keeps the last-token marginal, the probability of a complete two-token yield, and the distribution of the last three tokens. If the final child yields only two tokens, its preceding token comes from the penultimate child. Otherwise all three tokens come from the final child. This gives the exact distribution for each fixed grammar and a uniform start symbol, without enumerating sentences or sampling them.

We use $\eta=V$ throughout. Values of $F$ are chosen so $FV$ and $FV^2$ are integers; no rounding of the requested grammar density is needed. Faint dots and lines show individual grammar seeds; shaded bands show mean ± one standard deviation **across grammars**, not sampling uncertainty. Ten seeds are used per parameter pair. The proposed formula is treated as a prediction to check, rather than assumed to be exact.

In [ ]:
import matplotlib.pyplot as plt


def analytical_trigram_entropy(v, f):
    return np.log1p(v * f) + (1 - f) * np.log(2)


def exact_final_trigram_distribution(grammar, v, depth):
    if depth < 2:
        raise ValueError('Use depth >= 2 so every sentence has at least three tokens.')
    last = np.zeros((v, v))
    short = np.zeros((v, v, v))
    suffix = np.zeros((v, v, v, v))
    # At the bottom, a production yields either two or three terminals.
    for lhs, productions in grammar[f'layer{depth}'].items():
        parent = (int(lhs) - 1) % v
        total = sum(prob for _, prob in productions)
        for rhs, prob in productions:
            children = tuple((int(x) - 1) % v for x in rhs)
            weight = prob / total
            last[parent, children[-1]] += weight
            if len(children) == 2:
                short[(parent,) + children] += weight
            else:
                suffix[(parent,) + children] += weight
    for layer in range(depth - 1, 0, -1):
        transition = np.zeros((v, v))
        pair_transition = np.zeros((v, v, v))
        for lhs, productions in grammar[f'layer{layer}'].items():
            parent = (int(lhs) - 1) % v
            total = sum(prob for _, prob in productions)
            for rhs, prob in productions:
                previous, final = [(int(x) - 1) % v for x in rhs[-2:]]
                transition[parent, final] += prob / total
                pair_transition[parent, previous, final] += prob / total
        new_suffix = np.einsum('ij,jabc->iabc', transition, suffix, optimize=True)
        if np.any(short):
            new_suffix += np.einsum(
                'ijk,ja,kbc->iabc', pair_transition, last, short, optimize=True
            )
        last = transition @ last
        suffix = new_suffix
        short = np.zeros_like(short)  # Every higher-layer yield has >= 4 terminals.
    joint = suffix.mean(axis=0)  # Same uniform start-symbol prior as the sampler.
    np.testing.assert_allclose(joint.sum(), 1.0, atol=1e-12)
    return joint


# The entropy routine also accepts nonnegative probability masses as counts.
exact_joint = exact_final_trigram_distribution(grammar, V, DEPTH)
exact_entropy = conditional_entropy_from_counts(exact_joint)
print(f'Single grammar: sampled = {entropy_nats:.6f}, exact = {exact_entropy:.6f} nats')
print(f'Formula = {analytical_trigram_entropy(V, F):.6f} nats')

In [ ]:
# Edit these grids to extend the comparison (larger V uses more memory).
V_VALUES = [8, 16, 32, 64]
F_VALUES = [1/8, 1/4, 1/2, 3/4, 1.0]
GRAMMAR_SEEDS = list(range(12345678, 12345688))  # Ten independent grammars per (V, F).
SWEEP_DEPTH = 3

results = []
for v in V_VALUES:
    for f in F_VALUES:
        assert np.isclose(f * v, round(f * v))
        assert np.isclose(f * v**2, round(f * v**2))
        m2, m3 = round(f * v), round(f * v**2)
        eta = v
        prob3 = 1 / (m2 * eta + m3)
        prob2 = eta * prob3
        entropies = []
        for seed in GRAMMAR_SEEDS:
            rules, _ = generate_rules_double_new(
                v, v, m2, m3, prob2, prob3, SWEEP_DEPTH, seed
            )
            joint = exact_final_trigram_distribution(generate_grammar(rules), v, SWEEP_DEPTH)
            entropies.append(conditional_entropy_from_counts(joint))
            del rules, joint
        prediction = float(analytical_trigram_entropy(v, f))
        results.append(dict(V=v, F=f, numerical=np.mean(entropies),
                            std=np.std(entropies, ddof=1) if len(entropies) > 1 else 0.,
                            formula=prediction, entropies=entropies))
    print(f'Finished V = {v}')

print(' V      F    numerical       std    formula  num - formula (nats)')
for row in results:
    print(f"{row['V']:2d}  {row['F']:5.3f}    {row['numerical']:.6f}  "
          f"{row['std']:.6f}   {row['formula']:.6f}  "
          f"{row['numerical'] - row['formula']:+.6f}")

## Normalized comparison

The plots below divide entropies and their spread by $\log V$. The exact numerical curves end at 1 when $F=1$. Analytical curves (including the dotted $\log(1+VF)$ comparison) are hidden wherever their normalized values exceed 1. The residual panel shows the full normalized numerical-minus-formula difference. The table above remains in nats.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for index, v in enumerate(V_VALUES):
    color = colors[index % len(colors)]
    rows = [r for r in results if r['V'] == v]
    fs = np.array([r['F'] for r in rows])
    normalizer = np.log(v)
    numerical = np.array([r['numerical'] for r in rows]) / normalizer
    std = np.array([r['std'] for r in rows]) / normalizer
    prediction = np.array([r['formula'] for r in rows]) / normalizer
    seed_values = np.array([r['entropies'] for r in rows]) / normalizer
    for ax, center, values in [
        (axes[0], numerical, seed_values),
        (axes[1], numerical - prediction, seed_values - prediction[:, None]),
    ]:
        # Every column follows one grammar seed through the F grid.
        ax.plot(fs, values, color=color, linewidth=0.7, alpha=0.22,
                marker='.', markersize=4, zorder=1)
        ax.fill_between(fs, center - std, center + std,
                        color=color, alpha=0.18, linewidth=0, zorder=2)
        ax.plot(fs, center, 'o-', color=color, markersize=5,
                linewidth=1.8, zorder=3)
    dense_f = np.linspace(min(fs), max(fs), 200)
    formula_curve = analytical_trigram_entropy(v, dense_f) / normalizer
    axes[0].plot(dense_f, np.where(formula_curve <= 1, formula_curve, np.nan),
                 color=color, linestyle='--', linewidth=2, zorder=4)
    baseline_curve = np.log1p(v * dense_f) / normalizer
    axes[0].plot(dense_f, np.where(baseline_curve <= 1, baseline_curve, np.nan),
                 color=color, linestyle=':', linewidth=2, zorder=4)
axes[0].set_ylabel(r'$H(\mathrm{last}\mid\mathrm{previous\ two})/\log V$')
axes[0].set_title('Normalized entropy and analytical prediction')
axes[0].set_ylim(0, 1)
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_ylabel(r'$(H_{\mathrm{numerical}}-H_{\mathrm{formula}})/\log V$')
axes[1].set_title('Deviation and spread across grammar seeds')
color_handles = [Line2D([], [], color=colors[i % len(colors)], label=f'V={v}')
                 for i, v in enumerate(V_VALUES)]
style_handles = [
    Line2D([], [], color='gray', linestyle='--', label='Analytical formula'),
    Line2D([], [], color='gray', linestyle=':', label=r'$\log(1+VF)$'),
    Line2D([], [], color='gray', marker='o', label='Numerical mean'),
    Line2D([], [], color='gray', alpha=0.35, marker='.', linewidth=0.7,
           label=f'Individual seeds (n={len(GRAMMAR_SEEDS)})'),
    Patch(facecolor='gray', alpha=0.18, label='Mean ± 1 SD'),
]
for ax in axes:
    ax.set_xlabel('F')
    ax.grid(alpha=0.25)
axes[0].legend(handles=color_handles + style_handles, fontsize=8, loc='best')
axes[1].legend(handles=color_handles, fontsize=9)
plt.show()


## Interpretation

The prediction is $\log(1+VF)+(1-F)\log 2$. The table and residual plot quantify its agreement with the exact grammar entropy. Faint traces display all ten seeds and the shaded bands show one standard deviation across grammars; neither represents finite-sample estimation error.

At $F=1$, the exact entropy is $\log V$, whereas the expression gives $\log(1+V)>\log V$. Thus this prediction still cannot be exact across the full range of $F$.


## Finite-\(V\) prediction: hypergeometric candidate counts

This section compares the **exact trigram entropy of the generated varying-tree grammar** (computed above by recursion) with a more accurate finite-\(V\) prediction for a *simplified suffix model*. The model approximates the unnormalized final-trigram weights by \(W_{abc}=T_{abc}+B_{bc}\), with indicator variables for allowed ternary and binary right-hand sides, assuming equal effective suffix-topology contributions. It **does not** claim that the full varying-tree grammar has these exact weights.

Under **global sampling without replacement** of right-hand sides, let

\[
N_T\sim\mathrm{Hypergeom}(V^3,FV^3,V),\qquad
N_B\sim\mathrm{Hypergeom}(V^2,FV^2,V).
\]

The finite-size predictions (in nats) are

\[
L_{FT}^{\mathrm{finite}}=\frac{\mathbb{E}[N_T\ln N_T]}{VF},\qquad
L_{VT}^{\mathrm{finite}}=\frac{\mathbb{E}[(N_T+N_B)\ln(N_T+N_B)]}{2VF}-F\ln 2.
\]

In the second expression, counts for overlapping binary/ternary candidates have twice the weight, so their contributions are *not* treated as independent tokens. These are size-biased entropy averages: contexts occur with probability proportional to their total rule weight. We compare both the varying-tree prediction and its difference from the fixed-tree prediction. The asymptotic difference is \((1-F)\ln2\), which need not be accurate at small \(VF\).

In [ ]:
from scipy.stats import hypergeom
from scipy.special import xlogy


def finite_v_trigram_theory(v, f):
    """Finite-v suffix-model predictions in nats; f*v and f*v**2 must be integers."""
    k2, k3 = round(f * v), round(f * v**2)
    if not (1 <= k2 <= v and 1 <= k3 <= v**2):
        raise ValueError('Require 0 < f <= 1 and f*v, f*v**2 >= 1.')
    if not (np.isclose(k2, f * v) and np.isclose(k3, f * v**2)):
        raise ValueError('Choose f so f*v and f*v**2 are integers.')

    n = np.arange(v + 1)
    # Across all v parents: v*m3 ternary RHSs among v^3; v*m2 binary RHSs among v^2.
    p_t = hypergeom.pmf(n, v**3, v * k3, v)
    p_b = hypergeom.pmf(n, v**2, v * k2, v)
    p_z = np.convolve(p_t, p_b)
    z = np.arange(2 * v + 1)

    fixed = float(p_t @ xlogy(n, n) / (v * f))
    varying = float(p_z @ xlogy(z, z) / (2 * v * f) - f * np.log(2))
    return dict(finite_FT=fixed, finite_VT=varying,
                finite_excess=varying-fixed,
                large_v_excess=(1-f)*np.log(2),
                old_formula=np.log1p(v*f)+(1-f)*np.log(2))


for v in [8, 16, 32, 64]:
    p = finite_v_trigram_theory(v, 1/4)
    print(f"v={v:2d}  FT={p['finite_FT']:.6f}  VT={p['finite_VT']:.6f}  "
          f"excess={p['finite_excess']:.6f}  asymptotic={p['large_v_excess']:.6f}")

# Sanity check: with all RHSs allowed, both models predict uniform last token.
for v in [8, 16, 32, 64]:
    p = finite_v_trigram_theory(v, 1.0)
    np.testing.assert_allclose([p['finite_FT'], p['finite_VT']], np.log(v), atol=1e-10)


### Select the numerical data

By default, reuse `results` from the existing exact-grammar sweep above (which already includes \(V=64\)). If that sweep is prohibitively slow, switch `REUSE_EXISTING_SWEEP` to `False`: the code then generates a **targeted sweep at \(F=1/4\)** with a configurable number of independent grammar seeds. It uses your existing `exact_final_trigram_distribution`, rather than sampling sequences, so there is no finite-sample plug-in bias. At large \(V\), the exact recursion can nevertheless be expensive in CPU time and memory.

The numerical estimates are for the actual varying-tree grammar, whereas the finite-size formulas are for the simplified equal-topology-weight suffix model. Disagreement is therefore informative, not necessarily a bug.

In [ ]:
REUSE_EXISTING_SWEEP = True
TARGET_F = 1 / 4
TARGET_V_VALUES = [8, 16, 32, 64]
TARGET_SEEDS = [12345678, 12345679, 12345680]  # Increase for tighter grammar averages.

if REUSE_EXISTING_SWEEP:
    if 'results' not in globals():
        raise RuntimeError('Run the existing exact-grammar sweep (cell defining results), '
                           'or set REUSE_EXISTING_SWEEP=False.')
    comparison_results = results
else:
    comparison_results = []
    for v in TARGET_V_VALUES:
        f = TARGET_F
        m2, m3 = round(f*v), round(f*v**2)
        p3 = 1 / (m2*v + m3)
        p2 = v * p3
        entropies = []
        for seed in TARGET_SEEDS:
            rules, _ = generate_rules_double_new(
                v, v, m2, m3, p2, p3, DEPTH, seed
            )
            g = generate_grammar(rules)
            joint = exact_final_trigram_distribution(g, v, DEPTH)
            entropies.append(conditional_entropy_from_counts(joint))
            del rules, g, joint
        comparison_results.append(dict(
            V=v, F=f, numerical=float(np.mean(entropies)),
            std=float(np.std(entropies, ddof=1)) if len(entropies)>1 else 0.0,
            entropies=entropies,
        ))
        print(f'Done: V={v}, F={f}, grammars={len(entropies)}')

finite_comparison = [dict(r, **finite_v_trigram_theory(r['V'], r['F']))
                     for r in comparison_results]
finite_comparison.sort(key=lambda r: (r['F'], r['V']))

print(' V     F    exact VT    finite VT   old formula   exact - finite   grammars')
for r in finite_comparison:
    print(f"{r['V']:2d}  {r['F']:5.3f}  {r['numerical']:10.6f}  "
          f"{r['finite_VT']:10.6f}  {r['old_formula']:11.6f}  "
          f"{r['numerical'] - r['finite_VT']:+13.6f}  {len(r['entropies']):3d}")


### Compare theory and exact-grammar entropy

The left panel shows exact varying-tree trigram loss and both approximations as a function of \(V\) at the selected \(F\). The right panel displays the residuals in nats; the faint line is the old large-\(V\) approximation minus the finite-size prediction. Error bars represent **one standard deviation across grammar realizations**, not uncertainty from sequence sampling. For a loss in units of \(\ln V\), set `NORMALIZE = True`.

In [ ]:
PLOT_F = 1/4
NORMALIZE = False
plot_rows = sorted([r for r in finite_comparison
                    if np.isclose(r['F'], PLOT_F)], key=lambda r:r['V'])
if not plot_rows:
    raise ValueError('No data at selected PLOT_F.')
vv = np.array([r['V'] for r in plot_rows])
scale = np.log(vv) if NORMALIZE else np.ones(len(vv))
exact = np.array([r['numerical'] for r in plot_rows]) / scale
sd = np.array([r['std'] for r in plot_rows]) / scale
finite = np.array([r['finite_VT'] for r in plot_rows]) / scale
old = np.array([r['old_formula'] for r in plot_rows]) / scale

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2), constrained_layout=True)
ax[0].errorbar(vv, exact, yerr=sd, fmt='o-', capsize=3,
               label='Exact varying-tree grammar (mean ± SD)')
ax[0].plot(vv, finite, 's--', label='Finite-v suffix model')
ax[0].plot(vv, old, '^:', label=r'Old $\ln(1+VF)+(1-F)\ln 2$')
ax[0].set_ylabel(r'$H/\ln V$' if NORMALIZE else 'Trigram entropy (nats)')
ax[0].set_title(fr'Trigram loss, $F={PLOT_F:g}$')
ax[0].legend(fontsize=9)

ax[1].errorbar(vv, exact-finite, yerr=sd, fmt='o-', capsize=3,
               label='Exact minus finite-v theory')
ax[1].plot(vv, old-finite, '^:', label='Old formula minus finite-v theory')
ax[1].axhline(0, color='black', ls='--', lw=1)
ax[1].set_title('Residuals')
ax[1].set_ylabel('Normalized residual' if NORMALIZE else 'Residual (nats)')
ax[1].legend(fontsize=9)
for axis in ax:
    axis.set_xlabel(r'Vocabulary size $V$')
    axis.set_xticks(vv)
    axis.grid(alpha=0.25)
plt.show()


### Finite-size correction over the full density grid

This isolates what the finite-\(V\) calculation changes relative to the earlier approximation, across every \((V,F)\) available in `comparison_results`. It also shows whether the new approximation reduces the discrepancy with the actual grammar. The two curves need not coincide, because the simplified suffix model does not incorporate the full tree-topology weights.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), constrained_layout=True)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for i, v in enumerate(sorted({r['V'] for r in finite_comparison})):
    r_v = sorted((r for r in finite_comparison if r['V']==v), key=lambda r:r['F'])
    f = np.array([r['F'] for r in r_v])
    exact_minus_finite = np.array([r['numerical']-r['finite_VT'] for r in r_v])
    finite_minus_old = np.array([r['finite_VT']-r['old_formula'] for r in r_v])
    color = colors[i % len(colors)]
    axes[0].plot(f, finite_minus_old, 'o-', color=color, label=f'V={v}')
    axes[1].errorbar(f, exact_minus_finite,
                     yerr=[r['std'] for r in r_v],
                     fmt='o-', capsize=2, color=color, label=f'V={v}')
axes[0].set_title('Finite-v theory minus old approximation')
axes[1].set_title('Exact grammar minus finite-v theory')
for ax in axes:
    ax.axhline(0, color='black', ls='--', lw=1)
    ax.set_xlabel(r'Rule fraction $F$')
    ax.set_ylabel('Difference (nats)')
    ax.grid(alpha=.25)
    ax.legend(fontsize=9)
plt.show()
